# 05 Execution Assumptions

第五课目标：理解交易信号和真实成交之间的差异。重点是比较 `close-to-close` 和 `next-open` 两种执行假设。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from quant_trading.market_data import download_ohlcv
from quant_trading.moving_average import (
    add_moving_average_strategy,
    add_moving_average_strategy_next_open,
)
from quant_trading.validation import (
    compare_execution_assumptions,
    format_execution_comparison_table,
    plot_execution_comparison,
)

## 1. 两种执行假设

`close-to-close`：昨天信号决定今天持仓，收益用昨天收盘到今天收盘计算。这很常见，但隐含成交价比较理想。

`next-open`：今天收盘后生成信号，明天开盘执行，更接近真实交易。

In [ ]:
SYMBOL = "SPY"
SHORT_WINDOW = 10
LONG_WINDOW = 200
TRANSACTION_COST_BPS = 1.0

df = download_ohlcv(symbol=SYMBOL, start="2000-01-01", auto_adjust=True)
df.head()

## 2. close-to-close 模型

这个模型使用 `Close.pct_change()` 作为持仓收益。它适合快速研究，但不是最保守的执行假设。

In [ ]:
close_to_close = add_moving_average_strategy(
    df,
    short_window=SHORT_WINDOW,
    long_window=LONG_WINDOW,
    transaction_cost_bps=TRANSACTION_COST_BPS,
)
close_to_close[["Close", "signal", "position", "return", "strategy_return"]].dropna().head()

## 3. next-open 模型

这个模型用今天收盘后的信号，决定明天开盘后的持仓。持仓收益用 `Open.shift(-1) / Open - 1` 近似。

In [ ]:
next_open = add_moving_average_strategy_next_open(
    df,
    short_window=SHORT_WINDOW,
    long_window=LONG_WINDOW,
    transaction_cost_bps=TRANSACTION_COST_BPS,
)
next_open[["Open", "Close", "signal", "position", "open_to_next_open_return", "strategy_return"]].dropna().head()

## 4. 比较两种执行假设

真实研究中，不能只用一种乐观执行假设。至少要检查更保守、更接近真实成交的版本。

In [ ]:
comparison = compare_execution_assumptions(
    df,
    short_window=SHORT_WINDOW,
    long_window=LONG_WINDOW,
    transaction_cost_bps=TRANSACTION_COST_BPS,
)
print(format_execution_comparison_table(comparison))

In [ ]:
fig = plot_execution_comparison(close_to_close, next_open)
fig

## 5. 本课结论

执行假设会改变策略结果。以后回测必须写清楚：信号何时产生、何时下单、用什么价格成交、成本和滑点如何处理。